# Hidden Value: Phase 2 - Build the 2026 Drive Dataset

**Goal of this notebook:** turn 2026 NFL play-by-play into one clean row per offensive drive, and nothing else.

No Drive Quality Score, no charts, no scoring analysis. That is Phase 3 and later. This notebook stops at a validated `hidden_value_2026_drives.csv`.

Every transformation below cites the section of **Design Document v1.0 (locked Sept 10, 2026)** it comes from, so the code and the methodology stay in lock-step.

> This notebook pulls live data from nflverse, so it must run in an environment with internet access (your machine, Colab, etc.).

## 1. Setup

In [13]:
%pip install nflreadpy

Note: you may need to restart the kernel to use updated packages.


In [14]:
# nflreadpy is the current nflverse Python package.
# (nfl_data_py was deprecated in Sep 2025 in favor of nflreadpy.)
# First time only, uncomment to install:
# %pip install nflreadpy pandas numpy

import nflreadpy as nfl
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

print("nflreadpy", getattr(nfl, "__version__", "?"))
print("pandas", pd.__version__)

nflreadpy 0.1.5
pandas 2.3.3


## 2. Pull 2026 PBP

In [15]:
%pip install polars pyarrow

Note: you may need to restart the kernel to use updated packages.


In [16]:
# Design Doc Sec 3 (unit = one offensive drive) / Sec 4 (what counts as a drive).
# nflreadpy returns a Polars frame; convert to pandas.
raw = nfl.load_pbp(seasons=[2026])
try:
    pbp = raw.to_pandas(use_pyarrow_extension_array=False)
except Exception:
    pbp = raw.to_pandas()

print("rows (all plays):", len(pbp))
print("seasons:", sorted(pbp["season"].dropna().unique().tolist()))
print("season_types:", pbp["season_type"].dropna().unique().tolist())
print("weeks present:", sorted(pbp["week"].dropna().unique().tolist()))
print("games:", pbp["game_id"].nunique())

rows (all plays): 21191
seasons: [2026]
season_types: ['REG']
weeks present: [1, 2, 3, 4, 5, 6, 7, 8]
games: 121


In [17]:
# Design Doc Sec 21.5: regular season is the primary population.
pbp = pbp[pbp["season_type"] == "REG"].copy()
print("regular-season plays:", len(pbp))
print("regular-season games:", pbp["game_id"].nunique())
print("weeks:", sorted(pbp["week"].dropna().unique().tolist()))

regular-season plays: 21191
regular-season games: 121
weeks: [1, 2, 3, 4, 5, 6, 7, 8]


## 3. Raw data audit\n\nConfirm the columns we rely on exist, and look at the drive fields before trusting them.

In [18]:
needed = ["game_id","play_id","season","week","posteam","defteam",
          "fixed_drive","fixed_drive_result","down","ydstogo","yardline_100",
          "play_type","yards_gained","epa","success","sack","interception",
          "fumble_lost","penalty","penalty_team","qb_kneel","qb_spike","wp","qtr",
          "third_down_converted","third_down_failed","drive_start_yard_line"]
missing = [c for c in needed if c not in pbp.columns]
print("MISSING COLUMNS (handle before proceeding):", missing if missing else "none")

print("\nfixed_drive null share:", round(pbp["fixed_drive"].isna().mean(), 4))
print("\nfixed_drive_result values:")
print(pbp["fixed_drive_result"].value_counts(dropna=False))
print("\nplay_type values:")
print(pbp["play_type"].value_counts(dropna=False))
print("\nmissing-value share, key numeric cols:")
print(pbp[["epa","yardline_100","yards_gained","wp"]].isna().mean().round(4))

MISSING COLUMNS (handle before proceeding): none

fixed_drive null share: 0.0

fixed_drive_result values:
fixed_drive_result
Touchdown            6252
Punt                 5748
Field goal           4421
Turnover             1534
Turnover on downs    1353
End of half          1000
Missed field goal     776
Opp touchdown          99
Safety                  5
None                    3
Name: count, dtype: int64

play_type values:
play_type
pass           8576
run            6334
no_play        2213
kickoff        1248
punt            913
None            619
extra_point     558
field_goal      504
qb_kneel        181
qb_spike         45
Name: count, dtype: int64

missing-value share, key numeric cols:
epa             0.0114
yardline_100    0.0716
yards_gained    0.0302
wp              0.0057
dtype: float64


## 4 & 5. Build play-level flags, then one row per drive\n\nWe first tag every play with the exact flags the design doc defines, so the drive table is just sums of those flags. Then we aggregate.

In [19]:
# Work only on plays that belong to a drive.
d = pbp[pbp["fixed_drive"].notna()].copy()
d["drive_id"] = d["game_id"].astype(str) + "_" + d["fixed_drive"].astype(int).astype(str)

# make sure optional source columns exist so aggregation never crashes
for c in ["drive_start_yard_line","third_down_converted","third_down_failed"]:
    if c not in d.columns:
        d[c] = np.nan

def col(name, default=0.0):
    return d[name].fillna(default) if name in d.columns else pd.Series(default, index=d.index)

pt    = d["play_type"].fillna("")
yg    = col("yards_gained")
epa   = d["epa"] if "epa" in d.columns else pd.Series(np.nan, index=d.index)
kneel = col("qb_kneel"); spike = col("qb_spike")
sack  = col("sack");     pen   = col("penalty")
intc  = col("interception"); fl = col("fumble_lost")
wp    = d["wp"].fillna(0.5) if "wp" in d.columns else pd.Series(0.5, index=d.index)
qtr   = col("qtr")
penteam = d["penalty_team"] if "penalty_team" in d.columns else pd.Series(np.nan, index=d.index)

# Sec 5 + Sec 12: qualifying scrimmage plays = pass/run, excluding kneels & spikes.
d["is_qual"] = pt.isin(["pass","run"]) & (kneel != 1) & (spike != 1)
d["is_pass"] = d["is_qual"] & (pt == "pass")   # dropbacks incl. sacks
d["is_rush"] = d["is_qual"] & (pt == "run")    # designed runs incl. scrambles

# Sec 5: success = EPA > 0 (use nflfastR 'success' if present, else derive).
succ = d["success"].fillna(0) if "success" in d.columns else (epa > 0).astype(float)
d["succ_q"] = (d["is_qual"] & (succ == 1)).astype(int)

# Sec 5: EPA summed only over qualifying plays.
d["epa_q"] = np.where(d["is_qual"], epa.fillna(0.0), 0.0)

# Sec 6: explosive = run 10+ / pass 15+ (primary thresholds, locked).
d["expl"] = (d["is_qual"] & (((pt=="run") & (yg>=10)) | ((pt=="pass") & (yg>=15)))).astype(int)

# Sec 9: negative plays = sacks, runs for loss, offensive-penalty no-plays.
d["neg"] = ((sack==1) | ((pt=="run") & (yg<0)) |
            ((pt=="no_play") & (pen==1) & (penteam == d["posteam"]))).astype(int)

# Sec 5: third-down conversions / failures (nflfastR flags).
d["t3_conv"] = col("third_down_converted")
d["t3_fail"] = col("third_down_failed")

# Sec 9: turnovers and sacks.
d["turnover"] = ((intc==1) | (fl==1)).astype(int)
d["sack_i"]   = (sack==1).astype(int)

# Sec 11: garbage-time PLAY = 2nd half (qtr>=3) AND win prob <10% or >90%,
# on qualifying plays. The DRIVE flag is decided after aggregation (majority rule).
d["gt_q"] = (d["is_qual"] & (qtr>=3) & ((wp<0.10) | (wp>0.90))).astype(int)

# posteam/defteam carried only from qualifying plays so special teams can't set them.
d["posteam_q"] = np.where(d["is_qual"], d["posteam"], np.nan)
d["defteam_q"] = np.where(d["is_qual"], d["defteam"], np.nan)

print("play-level flags built on", len(d), "drive-eligible plays")

play-level flags built on 21191 drive-eligible plays


### 5. Aggregate to one row per drive

In [20]:
g = d.groupby("drive_id", sort=False)

drives = g.agg(
    game_id=("game_id","first"),
    season=("season","first"),
    week=("week","first"),
    posteam=("posteam_q","first"),
    defteam=("defteam_q","first"),
    fixed_drive=("fixed_drive","first"),
    drive_result=("fixed_drive_result","first"),
    drive_start_yard_line=("drive_start_yard_line","first"),
    qualifying_plays=("is_qual","sum"),
    pass_plays=("is_pass","sum"),
    rush_plays=("is_rush","sum"),
    success_plays=("succ_q","sum"),
    epa_sum=("epa_q","sum"),
    explosive_plays=("expl","sum"),
    third_down_conversions=("t3_conv","sum"),
    negative_plays=("neg","sum"),
    sacks=("sack_i","sum"),
    turnovers=("turnover","sum"),
    gt_qual_plays=("gt_q","sum"),
).reset_index()

t3 = g.agg(c=("t3_conv","sum"), f=("t3_fail","sum"))
drives["third_down_attempts"] = drives["drive_id"].map(t3["c"] + t3["f"])

# ---- FIELD POSITION + PENETRATION: scrimmage plays ONLY (Sec 7, Sec 8) ----
# nflfastR bundles the kickoff into the drive and it sits at the 35, which would
# wreck both drive start and the reached-40/20 flags. Compute these from qualifying
# scrimmage plays only, never kickoffs/PATs.
s = d[d["is_qual"]].copy()
s["end_yl"] = s["yardline_100"] - s["yards_gained"]      # ball position AFTER the play

start_yl = s.sort_values("play_id").groupby("drive_id")["yardline_100"].first()
snap_min = s.groupby("drive_id")["yardline_100"].min()
end_min  = s.groupby("drive_id")["end_yl"].min()
deepest  = pd.concat([snap_min, end_min], axis=1).min(axis=1)   # closest to opp end zone

drives["start_yardline_100"]   = drives["drive_id"].map(start_yl)
drives["deepest_yardline_100"] = drives["drive_id"].map(deepest)

# Sec 10 outcomes (td_drive is needed before the reached flags).
res = drives["drive_result"].fillna("")
drives["td_drive"] = (res == "Touchdown").astype(int)
drives["fg_drive"] = (res == "Field goal").astype(int)
drives["offensive_points"] = np.where(res=="Touchdown", 7,
                              np.where(res=="Field goal", 3, 0))
drives["net_points"] = (drives["offensive_points"]
                        - 7*(res=="Opp touchdown").astype(int)
                        - 2*(res=="Safety").astype(int))
drives["clock_expiration"] = res.str.startswith("End").astype(int)

# Sec 7: reached opp 40 / opp 20. A scoring drive counts; deepest scrimmage
# position captures a TD or a play that gains into the zone.
drives["reached_40"] = ((drives["deepest_yardline_100"] <= 40) | (drives["td_drive"]==1)).astype(int)
drives["reached_20"] = ((drives["deepest_yardline_100"] <= 20) | (drives["td_drive"]==1)).astype(int)

# Sec 11: garbage-time drive = majority of qualifying plays in garbage time.
drives["garbage_time"] = ((drives["gt_qual_plays"] > drives["qualifying_plays"]/2)
                          & (drives["qualifying_plays"] > 0)).astype(int)

print("drives before qualifying filter:", len(drives))
drives[["week","posteam","drive_result","drive_start_yard_line",
        "start_yardline_100","deepest_yardline_100","reached_40","reached_20",
        "offensive_points"]].head(18)

drives before qualifying filter: 2650


,week,posteam,drive_result,drive_start_yard_line,start_yardline_100,deepest_yardline_100,reached_40,reached_20,offensive_points
0,1,ARI,Touchdown,ARI 30,70.0,0.0,1,1,7
1,1,LAC,Touchdown,LAC 23,77.0,0.0,1,1,7
2,1,ARI,Missed field goal,ARI 30,70.0,36.0,1,0,0
3,1,LAC,Turnover,LAC 44,56.0,30.0,1,0,0
4,1,ARI,Field goal,ARI 34,66.0,9.0,1,1,3
5,1,LAC,Turnover on downs,LAC 21,65.0,3.0,1,1,0
6,1,ARI,Field goal,ARI 13,87.0,11.0,1,1,3
7,1,LAC,Punt,LAC 23,77.0,57.0,0,0,0
8,1,ARI,Field goal,ARI 10,90.0,14.0,1,1,3
9,1,LAC,Touchdown,LAC 40,12.0,0.0,1,1,7


### 4 (applied). Keep only qualifying offensive drives\n\nDrop special-teams-only possessions, victory formations (all kneels), and any zero-scrimmage-play drive. Short *real* drives are kept.

In [21]:
before = len(drives)
drop_mask = (drives["qualifying_plays"] == 0) | (drives["posteam"].isna())
print("dropping", int(drop_mask.sum()), "of", before,
      "drives (zero qualifying plays / no offense: ST-only, victory formation, empty)")
drives = drives[~drop_mask].copy()
print("qualifying offensive drives:", len(drives))

dropping 91 of 2650 drives (zero qualifying plays / no offense: ST-only, victory formation, empty)
qualifying offensive drives: 2559


## 6. Drive-table validation\n\nCheck the table is internally consistent before we trust a single number from it.

In [22]:
# 1. Outcome values are only what the design doc allows.
assert set(drives["offensive_points"].unique()).issubset({0,3,7}), "unexpected offensive_points"
print("offensive_points values OK:", sorted(drives["offensive_points"].unique()))

# 2. reached_20 implies reached_40 (20 <= 40).
print("reached_20-without-40 (should be 0):",
      int(((drives.reached_20==1) & (drives.reached_40==0)).sum()))

# 3. No drive is both TD and FG.
print("TD-and-FG drives (should be 0):",
      int(((drives.td_drive==1) & (drives.fg_drive==1)).sum()))

# 4. Every drive has exactly one offense.
print("drives missing posteam (should be 0):", int(drives["posteam"].isna().sum()))

# 5. Turnovers should line up with turnover-type endings.
print("\nturnovers>0 vs drive_result:")
print(pd.crosstab(drives["turnovers"]>0, drives["drive_result"]))

# 6. TD drives should end in a 'Touchdown' result.
print("\ntd_drive vs drive_result:")
print(pd.crosstab(drives["td_drive"], drives["drive_result"]))

# 7. Sample drives to eyeball.
cols = ["week","posteam","defteam","drive_result","start_yardline_100",
        "qualifying_plays","success_plays","epa_sum","explosive_plays",
        "third_down_conversions","third_down_attempts","turnovers",
        "reached_20","offensive_points","net_points","garbage_time","clock_expiration"]
drives[cols].head(15)

offensive_points values OK: [np.int64(0), np.int64(3), np.int64(7)]
reached_20-without-40 (should be 0): 0
TD-and-FG drives (should be 0): 0
drives missing posteam (should be 0): 0

turnovers>0 vs drive_result:


drive_result  End of half  Field goal  Missed field goal  Opp touchdown  Punt  Safety  Touchdown  Turnover  Turnover on downs
turnovers                                                                                                                    
False                 106         416                 81              2   901       2        599         0                147
True                    0           4                  1             13    10       0          4       273                  0

td_drive vs drive_result:
drive_result  End of half  Field goal  Missed field goal  Opp touchdown  Punt  Safety  Touchdown  Turnover  Turnover on downs
td_drive                                                                                                                     
0                     106         420                 82             15   911       2          0       273                147
1                       0           0                  0              0     0       0      

,week,posteam,defteam,drive_result,start_yardline_100,qualifying_plays,success_plays,epa_sum,explosive_plays,third_down_conversions,third_down_attempts,turnovers,reached_20,offensive_points,net_points,garbage_time,clock_expiration
0,1,ARI,LAC,Touchdown,70.0,11,7,5.700554,2,2.0,2.0,0,1,7,7,0,0
1,1,LAC,ARI,Touchdown,77.0,8,4,5.313758,3,1.0,1.0,0,1,7,7,0,0
2,1,ARI,LAC,Missed field goal,70.0,7,3,1.032609,1,0.0,1.0,0,0,0,0,0,0
3,1,LAC,ARI,Turnover,56.0,4,2,-4.188022,1,0.0,0.0,1,0,0,0,0,0
4,1,ARI,LAC,Field goal,66.0,11,6,2.966778,1,2.0,3.0,0,1,3,3,0,0
5,1,LAC,ARI,Turnover on downs,65.0,10,4,-2.089605,1,0.0,1.0,0,1,0,0,0,0
6,1,ARI,LAC,Field goal,87.0,7,5,1.218147,0,0.0,0.0,0,1,3,3,0,0
7,1,LAC,ARI,Punt,77.0,5,3,-1.029517,0,0.0,1.0,0,0,0,0,0,0
8,1,ARI,LAC,Field goal,90.0,9,4,2.344937,3,0.0,1.0,0,1,3,3,0,0
9,1,LAC,ARI,Touchdown,12.0,3,1,1.600970,0,1.0,1.0,0,1,7,7,0,0


In [23]:
# Sec 15 (ranking gate) status: how close is each team to 40 non-garbage drives?
per_team = (drives[drives["garbage_time"]==0]
            .groupby("posteam").size().sort_values(ascending=False))
print("drives per team (garbage time excluded). 40 needed before any rankings:")
print(per_team)

drives per team (garbage time excluded). 40 needed before any rankings:
posteam
SEA    80
DEN    75
TB     75
PHI    75
CLE    73
ARI    72
PIT    72
IND    72
BAL    70
LA     69
WAS    68
NYJ    68
HOU    67
ATL    67
MIN    67
TEN    66
CHI    66
LAC    65
GB     65
NYG    64
BUF    64
LV     64
NE     62
DAL    62
DET    59
KC     58
CAR    57
NO     56
CIN    55
JAX    55
SF     51
MIA    50
dtype: int64


## 7. Export\n\nThis CSV is the Phase 2 deliverable and the input to Phase 3.

In [24]:
out = r"C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football\hidden_value_2026_drives.csv"
drives.to_csv(out, index=False)
print("saved", out, "-", len(drives), "drives x", drives.shape[1], "columns")
print("columns:", list(drives.columns))

saved C:\Users\brade\OneDrive - Worcester Polytechnic Institute (wpi.edu)\Desktop\Football\hidden_value_2026_drives.csv - 2559 drives x 31 columns
columns: ['drive_id', 'game_id', 'season', 'week', 'posteam', 'defteam', 'fixed_drive', 'drive_result', 'drive_start_yard_line', 'qualifying_plays', 'pass_plays', 'rush_plays', 'success_plays', 'epa_sum', 'explosive_plays', 'third_down_conversions', 'negative_plays', 'sacks', 'turnovers', 'gt_qual_plays', 'third_down_attempts', 'start_yardline_100', 'deepest_yardline_100', 'td_drive', 'fg_drive', 'offensive_points', 'net_points', 'clock_expiration', 'reached_40', 'reached_20', 'garbage_time']


## Next: Phase 3 (do NOT start yet)

Before any scoring or Drive Quality work, we inspect the table above:
- Do the drive counts, results, and field positions look real?
- Do the validation cross-tabs come back clean?
- Are we past the 40-drive gate for rankings, or still in small-sample territory (expected this early in the season)?

Only once this drive table is trusted do we move to exploration and metric relationships. The methodology is already locked; Phase 3 just applies it.